# tidypr: build the master dataset (free teacher on Kaggle)

**Pipeline:** public review datasets → keyword prefilter → **Qwen2.5-Coder-7B teacher** labels each
(hunk + human comment) → spot-check → merge with real clean hunks + synthetic → upload to HF (private).

Kaggle settings: **GPU T4 x1**, **Internet On**, secret **`HF_TOKEN`** (write token, for the upload).

1. Run with `LIMIT = 100` first (~15 min incl. installs) and look at the spot-check cell.
2. If labels look sensible, set `LIMIT = 0` and **Save & Run All (Commit)**. Rough guess: 1–3 h for ~10k candidates.

In [ ]:
# ---- config --------------------------------------------------------------------------
GIT_REPO      = "https://github.com/thebunnyweb/tidypr"
AIDEV         = 6000     # max AIDev candidates (humans reviewing AI-agent PRs)
CODEREVIEWER  = 6000     # max CodeReviewer candidates (9 languages)
CLEAN         = 6000     # max real clean hunks (CodeReviewer y=0)
SYNTHETIC     = 3000     # v2 synthetic clean/decoy/slop (Python)
LIMIT         = 100      # teacher smoke test; 0 = label everything
TEACHER       = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
HF_DATA_REPO  = "bunnyweb/tidypr-data"   # private dataset repo for the result ("" = don't upload)

In [ ]:
import os, json, subprocess, glob, random
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = "/kaggle/working/tidypr"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "-q", GIT_REPO, WORK], check=True)
os.chdir(WORK)
!git log --oneline -1
!pip install -q -e . pandas pyarrow huggingface_hub datasets
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception as e:
    print("no HF_TOKEN secret: upload will be skipped", e); HF_DATA_REPO = ""

## 1. Collect candidates (CPU)

In [ ]:
!python data/real/collect.py --peek --aidev 3 --codereviewer 3 --clean 3

In [ ]:
!python data/real/collect.py --out data/real/candidates.jsonl --clean-out data/real/clean.jsonl \
    --aidev {AIDEV} --codereviewer {CODEREVIEWER} --clean {CLEAN}
!wc -l data/real/candidates.jsonl data/real/clean.jsonl

## 2. Teacher labelling (GPU)

In [ ]:
!pip install -q vllm 2>&1 | tail -2
lim = f"--limit {LIMIT}" if LIMIT else ""
rc = os.system(f"python data/real/teacher.py --candidates data/real/candidates.jsonl --out data/real/labeled.jsonl "
               f"--backend vllm --model {TEACHER} {lim}")
if rc != 0:
    print("vLLM failed; falling back to transformers (slower)")
    !pip install -q bitsandbytes accelerate
    os.system(f"python data/real/teacher.py --candidates data/real/candidates.jsonl --out data/real/labeled.jsonl "
              f"--backend hf --model Qwen/Qwen2.5-Coder-7B-Instruct --batch 32 {lim}")

## 3. Spot-check: read these. Do the labels match what the reviewer meant?

In [ ]:
rows = [json.loads(l) for l in open("data/real/labeled.jsonl")]
from collections import Counter
print(Counter(r["label"] for r in rows), Counter(r["finding"]["category"] for r in rows if r["finding"]))
random.seed(0)
for r in random.sample([r for r in rows if r["label"] == "slop"], k=min(8, sum(r["label"] == "slop" for r in rows))):
    print("=" * 100, f"\n{r['source']} | {r['path']} | anchor {r['anchor']}")
    print("\n".join(f"{n:>5} {k} {t}" for n, k, t in r["hunk"][-12:]))
    print("REVIEWER:", r["comment"][:300]); print("TEACHER :", r["finding"])
print("\n\n#### some rejected (not_slop) comments:")
for r in [r for r in rows if r["label"] == "not_slop"][:6]:
    print(" -", r["comment"][:160].replace("\n", " "))

## 4. Merge into the master dataset + upload

In [ ]:
import sysconfig
STDLIB = sysconfig.get_paths()["stdlib"]
!python data/real/merge.py --out data/master --labeled data/real/labeled.jsonl --clean data/real/clean.jsonl \
    --synthetic {SYNTHETIC} --source "{STDLIB}={SYNTHETIC}"
!ls -lh data/master

In [ ]:
if HF_DATA_REPO and not LIMIT:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(HF_DATA_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path="data/master", repo_id=HF_DATA_REPO, repo_type="dataset", path_in_repo="master")
    api.upload_folder(folder_path="data/real", repo_id=HF_DATA_REPO, repo_type="dataset", path_in_repo="real",
                      allow_patterns=["*.jsonl"])
    print(f"uploaded → https://huggingface.co/datasets/{HF_DATA_REPO}")
else:
    print("smoke run (LIMIT set) or no HF repo: not uploading. Files are in /kaggle/working/tidypr/data/")
!cd data && zip -qr /kaggle/working/tidypr-master.zip master && ls -lh /kaggle/working/*.zip